<img src='./img/mqtt-banner.png'></img>

# MQTT Lab

In this lab, you'll learn about MQTT, a messaging protocol for the Internet of Things. MQTT is a lightweight publish/subscribe messaging protocol that is ideal for connecting small devices over the internet. The information below is condensed from https://mqtt.org/ and https://www.hivemq.com/mqtt/, you can always refer to those websites for more information.

## MQTT's architecture

MQTT has a publish/subscribe architecture in which clients connect to a central broker. Clients publish messages to the broker on a topic. Clients may also subscribe to topics of interest. The broker then forwards the published messages to the subscribers.

<img src='./img/mqtt-architecture.png'></img>

## Features of MQTT

MQTT was made with Internet of Things devices in mind, this means many resource constrained (small battery, low-power processors, small amount of memory) devices connected via unreliable low-bandwidth networks.

It has the following features:
* **Lightweight and Efficient**: MQTT clients require minimal resources, so they run on small microcontrollers. Message headers are also small to optimize network bandwidth.
* **Bi-directional Communications**
  : MQTT allows for device-to-cloud and cloud-to-device messaging. It supports easy broadcasting of information to a group of clients.
* **Scalability**: MQTT scales to millions of devices.
* **Reliable Delivery**: MQTT offers 3 quality of service levels, determining how often a single message may arrive at the broker and/or client. Level 0 - at most once, level 1 - at least once, level 2- exactly once.
* **Supports unreliable networks**: MQTT supports persistent sessions reducing reconnection time on unreliable networks.
* **Secure**: MQTT supports TLS encryption, and authentication of clients.

In the following exercises, we'll explore these features further.

## Connecting to the EMQX broker...

MQTT needs a broker to which clients can connect. Many such brokers exist, e.g. EMQX, HiveMQ, Eclipse Mosquitto. You'll find many of these brokers here: https://mqtt.org/software/ . Some brokers you can host yourself, others are hosted for you.
For our exercises, we have set up an EMQX broker to which you can connect. The broker's address is `ed1fe6fe.ala.eu-central-1.emqxsl.com`.

To connect, you should supply your username and password, as given in the users.csv file.
This should normally be:

* **username**: `firstname-lastname` (note: special characters such as the umlaut, accents, etc. have been replaced with the plain letter)
* **password**: `bip-mqtt-lab-2026`

To connect to the broker, we need a client to set up the connection. There are:
* Standalone clients, e.g. the **mosquitto_pub** and **mosquitto_sub** programs, as provided by the Eclipse Mosquitto project, [MQTTX](https://mqttx.app/), [MQTT-CLI](https://github.com/hivemq/mqtt-cli) and many more.
* Client libraries you can use in your favourite programming language. A popular client library is [Paho](https://github.com/eclipse-paho), which is available for C/C++, Python, Go, Java, Javascript and .Net

We'll use both the standalone clients [mosquitto_pub](https://mosquitto.org/man/mosquitto_pub-1.html) and [mosquitto_sub](https://mosquitto.org/man/mosquitto_sub-1.html), and [Paho for Python](https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html) in these exercises. We start out with the standalone clients, when you read this text, you're expected to execute the commands in your own terminal.


## ... with a standalone client

Because `mosquitto_sub` and `mosquitto_pub` are command line programs to be executed in a terminal, we'll open two terminals. You do so, by clicking on the new launcher button in Jupyter Lab, or by opening a new tab in Jupyter Lab, then selecting `Terminal` in the `Other` section. 

<img src='./img/new-launcher.png'></img>

Do this twice and place these terminals next to or on top of each other for easier readability.

<img src='./img/terminals.png'></img>

The first command sets up a client that subscribes to the topic "bip/mqtt-lab/test".

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab -v`

The options are:
* -h the host address of the MQTT broker.
* -p the port to connect to on the host. Typically MQTT uses port 1883 for unencrypted connections and 8883 for TLS encrypted connections
* -u the username to authenticate with - replace `<username>` with your username
* -P the password for that user
* -t to specify the topic to subscribe to
* -v to enable verbose output, this is optional, but it prints the topic on which a message was received alongside the message.

Copy and paste this command in one of the Terminals and execute it by pressing Enter. As you will see, no output appears yet, but the client remains idle, waiting for any messages to  arrive.

Let's now send a message to our subscriber with the other terminal. In it, copy and paste the following mosquitto_pub command:

`mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab -m "Hello World"`

The first options you already know. Replace the `<username>` again with your own. The final option:
`-m` specifies the message to be sent. In this case, we sent the plain string `"Hello world"`

If you did these steps correctly, the `"Hello world"` message should have arrived at the subscriber. Your terminals should look something like the image below. You're now ready to explorer some more features of MQTT.

<img src='./img/terminals-hello-world.png'></img>

### A note about permissions

The users created for this lab only have permission to write to the base topic `bip/mqtt-lab` and any subtopic (e.g. `bip/mqtt-lab/subtopic` or `bip/mqtt-lab/subtopic/another-subtopic` and so forth). Writing/subscribing to other topics will be rejected by the broker.


### Topic Filtering

At the subscriber's end, it might be useful to subscribe to multiple topics. To this end, MQTT has wildcard topic matching with the `+` and `#` characters.
Say, for example, you have a smart home, with temperature and humidity sensors on three floors, whose value is published on the following topics:

* `sensors/ground-floor/temperature`
* `sensors/ground-floor/humidity/inside`
* `sensors/ground-floor/humidity/outside`
* `sensors/first-floor/temperature`
* `sensors/second-floor/temperature`

Your application might be interested in all the temperature readings, regardless of the floor. You can subscribe to all of them at the same time by using the `+` wildcard, which matches a single topic level, by subscribing to:

`sensors/+/temperature`

Say that you are additionally interested in the humidity sensors, then you can subscribe to:

`sensors/+/+`

The `+` wildcard matches only a single level, that does mean you'll not get messages on the `inside` and `outside` topics for the humidity sensor. To also get those readings, you can use the `#` wildcard, which matches the current and any following levels. Hence, it is only used at the end of a subscription.

Since the humidity topics go 4 levels deep, whereas the temperature topic is only 3 levels deep, if you want to subscribe to all sensor readings, you can do so with the `#` wildcard as follows:

`sensors/#`

Say you only want sensors on the ground floor, you'd use

`sensors/ground-floor/#`

In either case, you'll be subscribed to the entire depth of the sensors, and receive the temperature messages, and the inside/outside humidity messages.

## Topic Filtering Exercise

say we have the following topic list, where we have two cranes, `crane-0` and `crane-1` who publish the position information of their cart and hoist:

* `bip/mqtt-lab/crane-0/hoist/x-pos`
* `bip/mqtt-lab/crane-0/hoist/y-pos`
* `bip/mqtt-lab/crane-0/cart/x-pos`
* `bip/mqtt-lab/crane-0/cart/y-pos`
* `bip/mqtt-lab/crane-1/hoist/x-pos`
* `bip/mqtt-lab/crane-1/hoist/y-pos`
* `bip/mqtt-lab/crane-1/cart/x-pos`
* `bip/mqtt-lab/crane-1/cart/y-pos`

Write the following filters:

* **hoist position filter**: a filter that subscribes to the hoist positions of all cranes
* **crane-1 filter**: a filter that subscribes to all the positions of crane-1
* **y-position filter**: a filter that subscribes to the y-position of all cranes and all crane parts

In [ ]:
# TODO complete these filters such that they filter the topics as described above. These will be graded
hoist_position_filter = 'fill/in/your/filter/+/#'
crane_1_filter = 'fill/in/your/filter/+/#'
y_position_filter = 'fill/in/your/filter/+/#'

### BEGIN SOLUTION
# possible solution:
# options with # also work.
hoist_position_filter = 'bip/mqtt-lab/+/hoist/+'
crane_1_filter = 'bip/mqtt-lab/crane-1/+/+'
y_position_filter = 'bip/mqtt-lab/+/+/y-pos'
### END SOLUTION

In [ ]:
# This cell is used for grading, please ignore it.
### BEGIN HIDDEN TESTS
assert hoist_position_filter == 'bip/mqtt-lab/+/hoist/+' or \
    hoist_position_filter == 'bip/mqtt-lab/+/hoist/#'
assert crane_1_filter == 'bip/mqtt-lab/crane-1/+/+' or \
    crane_1_filter == 'bip/mqtt-lab/crane-1/+/#' or \
    crane_1_filter == 'bip/mqtt-lab/crane-1/#'
assert y_position_filter == 'bip/mqtt-lab/+/+/y-pos'
### END HIDDEN TESTS

To verify if the filters work, set up a subscription with **mosquitto_sub** in the terminal, you'll need three terminals if you want to test every subscription at once. Then run the cell below, which transmits readings to the various topics.

In [ ]:
# run this cell to transmit data to the crane topics.
# then observe if the expected messages arrive on the subscribers.
# each topic receives one position update.
from helper_functions import publish_crane_position_nb
publish_crane_position_nb()

Connected to MQTT Broker!
Message '88' published on topic 'bip/mqtt-lab/crane-0/hoist/x-pos'
Message '0' published on topic 'bip/mqtt-lab/crane-0/hoist/y-pos'
Message '35' published on topic 'bip/mqtt-lab/crane-0/cart/x-pos'
Message '59' published on topic 'bip/mqtt-lab/crane-0/cart/y-pos'
Message '36' published on topic 'bip/mqtt-lab/crane-1/hoist/x-pos'
Message '67' published on topic 'bip/mqtt-lab/crane-1/hoist/y-pos'
Message '92' published on topic 'bip/mqtt-lab/crane-1/cart/x-pos'
Message '23' published on topic 'bip/mqtt-lab/crane-1/cart/y-pos'
Succesfully disconnected


### Quality of Service

MQTT also supports three quality of service levels. These levels define the delivery guarantees of a specific message.

* QoS 0: At most once (message arrives 0 or 1 times)
* QoS 1: At least once (message arrives 1 or more times)
* QoS 2: Exactly once (message arrives exactly 1 time)

The quality of service can be set by both the publisher and the subscriber.

* When a publisher sets the QoS, it is in effect for the packets sent between the publisher and the broker.
* When a subscriber sets the Qos, it is in effect for the packets sent from the broker to the subscriber.

#### Downgrading of QoS level

When looking at the QoS of a packet as a whole (i.e. going from publisher all the way to subscriber), the fact that both publisher and subscriber set their preferred QoS level means we can have a downgrade of QoS at the "packet as whole" level. For example, a publisher publishes with QoS 2 to the broker, whilst a subscriber is subscribed with QoS 1. Any packet may then arrive 1 or more times at the subscriber (QoS 1), so the "packet as a whole" QoS is downgraded from 2 to 1.

This does not work the other way around, that is, QoS is never upgraded. If a publisher publishes with QoS 0 and a subscriber subscribes with QoS 2, whether the packet arrives exactly once at the subscriber merely depends on whether the packet reaches the broker on the QoS 0 link from the publisher. The same holds true for other combinations (e.g. pub 0, sub 1 or pub 1 sub 2).

#### Mosquitto_pub and _sub

In `mosquitto_pub` and `mosquitto_sub`, the QoS level can be set with the `-q` or `--qos` option. So far we did not set this option, so the default of QoS 0 was used.

While it's difficult to really show packets being lost, we can enable the debug option with `-d`, which allows you to see the entire message exchange used to transmit the message.

We'll demonstrate for the three levels of QoS, please verify these outputs in your own terminal.

#### QoS 0 publish/subscribe

Subscriber command: `mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -v -q 0 -d`

Publisher command: `mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab -m 100 -q 0 -d`

Output: 

<img src='./img/pubsub-qos0-output.png'></img>

On both the subscriber and publisher we first see the connection to the broker being established with the `CONNECT` and  `CONNACK` (Connection Acknowledged) messages.

After connecting, the subscriber also subscribes to the topic and receives an acknowledgement with the `SUBSCRIBE` and `SUBACK` messages.

Next, the publisher publishes the message with `PUBLISH`. Becuase QoS 0 is used, this is just a 'one shot' publish. Then it disconnects with `DISCONNECT`.

On the subscriber side, we see the publish being received with `received PUBLISH`, again just a 'one shot' publish from the broker this time.

If the message is lost somewhere along the way, we'd never know.

#### QoS 1 publish/subscribe

Subscriber command: `mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -v -q 1 -d`

Publisher command: `mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab -m 100 -q 1 -d`

Output: 

<img src='./img/pubsub-qos1-output.png'></img>

The broker connection setup and the subscription are the same as before.

The publisher publishes the message again with `PUBLISH`, and now keeps that message in memory while it waits for the broker to confirm the reception with a `PUBACK` message. If the `PUBACK` message is not received before a timeout (because the original `PUBLISH` got lost, or because the `PUBACK` never made it all the way back), the publisher will retransmit the message, and wait for the `PUBACK` again. If the retransmission occurs because the `PUBACK` message got lost, the chance exists that the receiver gets the same message more than once, hence the 'at least once' operation of QoS 1.

At the receiver side, we see the same thing.

#### QoS 2 publish/subscribe


Subscriber command: `mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -v -q 2 -d`

Publisher command: `mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab -m 100 -q 2 -d`

Output: 

<img src='./img/pubsub-qos2-output.png'></img>

The broker connection setup and the subscription are the same as before.

Publishing now takes a four-part handshake: the publisher publishes the message again with `PUBLISH`, and keeps the message in memory while it waits for the sender to confirm reception with `PUBREC` (Publish Received) message. As long as `PUBREC` is not received within a timeout, the publisher will keep on republishing the packet. Once the `PUBREC` is received, the publisher can safely discard the stored packet. The publisher replies with `PUBREL` (Publish Release) indicating the message can be released. Upon reception, the broker confirms with `PUBCOMP` (Publish Complete). If the `PUBCOMP` message does not arrive in time, the publisher retransmits the `PUBREL`. If the reason `PUBCOMP` did not arrive was because the original `PUBREL` was lost, everything proceeds as normal. If the original `PUBREL` was received, but `PUBCOMP` was lost, the receiver will have already deleted the packet identifier from its local session state. If it then receives `PUBREL` it will respond with `PUBCOMP` with a flag set to indicate `Packet Identifier not found`. This tells the publisher there is a mismatch between its state and the receiver's state, and that the publish is completed.

Most important to note is that this entire process is because the MQTT session is stored at the application layer.

## QoS Downgrading

To demonstrate the downgrading of QoS, we subscribe with QoS 1 and publish with QoS 2:

Subscriber command: `mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -v -q 1 -d`

Publisher command: `mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab -m 100 -q 2 -d`

Output: 

<img src='./img/downgraded-qos.png'></img>

We see that the publishing occurs with QoS 2, while the transmission from the broker to the subscriber uses QoS 1, which is what we called a downgrade of QoS earlier on.

### Persistent Sessions

If a device were to lose its connection to the MQTT broker, it would have to set up a new connection and resubscribe to all the topics it was previously subscribed to. Because this subscription process might be burdensome for resource constrained devices, MQTT has the option of persistent sessions.

During the initial connection, a client can request a persistent session with the broker. It must provide a unique ID  that it will reuse when reconnecting to the broker.

A persistent session has the broker store:

* Info about the existence of the session
* The client's subscriptions
* Flow of messages in QoS 1 and QoS 2 (by keeping track of unacknowledged messages)
* All QoS 1 and QoS 2 messages the client missed whilst offline
* All QoS 2 messages received from the client that are awaiting complete acknowledgement

In `mosquitto_pub/sub` we'd have to use the additional flag `-c` or `--disable-clean-session` and provide a unique identifier with `-i` or `--id` to persist a session.

### Demonstration

Open two terminals and setup:

A regular subscriber with: 

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <user-name> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -v -q 2 -d`

A subscriber with a persistent connection, use for example your username also as id:

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <user-name> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -v -q 2 -d -c -i <user-name>`

After setting up the terminals, stop the process in the second terminal and run the following publish command:

`mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u joost-mertens -P bip-mqtt-lab-2026 -t bip/mqtt-lab -m "this message will be persisted" -q 2 -d`

So far, you should've seen the messages appear on the regular subscriber.

Now reconnect the subscriber with the same id you used previously with the command: `mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <user-name> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/some-other-topic -v -q 2 -d -c -i <user-name>`

You should see the stored messages arrive, as the broker stored them for us, and we were subscribed with QoS 2. (small note: mosquitto_sub forces you to provide a topic to subscribe to, so we just set it to `some-other-topic` that we didn't get a message on to show that that subscription to the crane topics was in fact retained in the session.)

Ouput below:

<img src='./img/persistent-sessions-demo.png'></img>

Finally, to end the persistent session, open a new connection without the `-c` flag but with the same id in the `-i` flag.

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <user-name> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/some-other-topic -v -q 2 -d -i <user-name>`


### Message Retention

Messages normally arrive only to currently subscribed clients. If you want a message to arrive to clients that subscribe after the message has been published, you can use MQTT's message retention option.

When you publish a message with message retention enabled, the broker stores the last message (only the last!) message published on that topic. Any client subscribing to the topic at a later point in time will immediately receive the retained message when they subscribe.


Let's test this, first we publish a retained message with **mosquitto_pub**:

`mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/retained-topic -m "this message will be retained" -q 2 -d -r`

Now let's connect a subscriber with **mosquitto_sub**:

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/retained-topic -q 2 -d`

Expected output:

<img src='./img/message-retention-demo.png'></img>

To delete the retained message from the broker, you publish a retained message with a zero-byte (empty) payload to the topic that stored the retained message.

Let's test this by deleting the previously retained message:

`mosquitto_pub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/retained-topic -m "" -q 2 -d -r`

(alternatively to writing `-m ""` you can also use the option `-n` or `--null-message` which also sends a null message)
And then subscribe with a subscriber:

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/retained-topic -q 2 -d`

As expected: the retained message was deleted since the subscriber did not receive any message upon subscribing.

Expected output:

<img src='./img/message-retention-remove-retained.png'></img>

## Last Will and Testament

Last Will and Testament is an MQTT feature that allows a client to specify a message that the broker should publish when the client disconnects unexpectedly. This feature is particularly useful when a client wants to notify others of their (unexpected) unavailability.

In **mosquitto_pub/sub**, the last will and testament message can be configured with the options:

* --will-payload
* --will-qos
* --will-retain
* --will-topic

Let's try it out using two terminals

In the first one, we'll set up a regular subscriber:

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -q 2 -d`

In he second one, we'll set up a regular subscriber with las will: 

`mosquitto_sub -h ed1fe6fe.ala.eu-central-1.emqxsl.com -p 8883 -u <username> -P bip-mqtt-lab-2026 -t bip/mqtt-lab/# -q 2 -d --will-payload "this is my last will" --will-qos 2 --will-topic bip/mqtt-lab/last-will-demo`

Preferably, when creating those two processes, wait one minute, that makes it easy to identify the two processes later on.

Next, we'll kill the second process with the kill command. (Note: we must really kill it, if we send the regular terminate, e.g. with Ctrl+Z, then the client disconnects gracefully, which is what we don't want)

First, get the process id's of the two **mosquitto_sub** processes with

`ps aux | grep mosquitto`

Then select the process id of the one that was created last (should be the second entry), copy the process id and enter the command

`kill -9 <copied-pid>`

This should ungracefully terminate the process and associated connections, after which the broker should publish the last will message, which you see in the still running subscriber.

Expected output is shown below:

<img src='./img/last-will-demo.png'></img>



## Break

Congratulations, you've so far mastered the basic feature set as present in MQTT 3.1.1 (nowadays, MQTT 5 is the latest version, with some tweaks and improvements, but the foundation remains the same, hence we won't be looking into those specific features in this lab). Take a small break before commencing the next section.

## Connecting with Paho

The next step is to look at how you can write a Python client using the Paho-mqtt library. To do so, we'll move over to the mqtt-lab-paho.ipynb notebook.